# P3: GRPO Training
**ATRD — Adaptive Test-Time Reasoning Distillation**

Phase 3: Group Relative Policy Optimization (GRPO) training loop with PRM rewards

- Model: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16` + SFT adapter
- Method: Reinforcement learning via GRPO
- Deliverable: GRPO-trained LoRA adapter

In [ ]:
# Cell 1: Imports and Reproducibility Setup
import random
import numpy as np
import torch
import os, sys, json, re
from pathlib import Path
from dataclasses import dataclass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Configuration
@dataclass(frozen=True)
class Phase3Config:
    BASE_MODEL: str = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16"
    SFT_CHECKPOINT: str = "/kaggle/working/checkpoints/sft/final_adapter" # P2 adapter output location
    GRPO_CONFIG: str = "configs/base_grpo.json"
    GROUP_SIZE: int = 8
    KL_PENALTY: float = 0.001
    LEARNING_RATE: float = 5e-6
    MAX_STEPS: int = 500
    OUTPUT_DIR: Path = Path("/kaggle/working/checkpoints/grpo")
    LOG_DIR: Path = Path("/kaggle/working/logs")

config = Phase3Config()
print(f"GRPO configured. Learning rate: {config.LEARNING_RATE} | Target steps: {config.MAX_STEPS}")

In [ ]:
# Cell 3: Load SFT Model + LoRA
sys.path.append('.') # Add workspace root to sys.path
from src.models.loader import ModelLoader, setup_blackwell_optimizations
from peft import PeftModel

setup_blackwell_optimizations()
loader = ModelLoader("configs/competition_params.json")
tokenizer = loader.load_tokenizer()

try:
    base_model = loader.load_model(quantize=True)
    if Path(config.SFT_CHECKPOINT).exists():
        model = PeftModel.from_pretrained(base_model, config.SFT_CHECKPOINT, is_trainable=True)
    else:
        print("SFT adapter checkpoint not found. Creating a fresh Lora config for GRPO...")
        from src.models.lora_config import create_lora_config
        from peft import get_peft_model
        lora_config = create_lora_config("configs/base_lora.json")
        model = get_peft_model(base_model, lora_config)
        
    model.print_trainable_parameters()
    loader.enable_gradient_checkpointing(model)
    print("Model and LoRA adapter prepared.")
except Exception as e:
    print(f"Skipped actual loading (running outside GPU environment): {e}")
    model = None
    base_model = None

In [ ]:
# Cell 4: Setup PRM Scorer
from src.training.grpo_trainer import GRPOTrainerWrapper

if model is not None:
    trainer = GRPOTrainerWrapper(
        model=model,
        tokenizer=tokenizer,
        output_dir=str(config.OUTPUT_DIR)
    )
    reward_fn = trainer.create_reward_function(tolerance=0.01)
    print("Reward function created with format + correctness + redundancy components")
else:
    trainer = None
    reward_fn = None
    print("Wrapper skipped (no model loaded).")

In [ ]:
# Cell 5: Load GRPO Training Data
from datasets import load_dataset
from pathlib import Path

dataset_file = Path("/kaggle/working/final_train_dataset.jsonl")
if dataset_file.exists():
    dataset = load_dataset("json", data_files=str(dataset_file))["train"]
else:
    print("dataset not found. Generating mock prompts for training flow validation...")
    from datasets import Dataset
    dummy_prompts = [
        {"question": "Solve for x: 3x = 9", "answer": "3"}
        for _ in range(20)
    ]
    dataset = Dataset.from_list(dummy_prompts)

grpo_train = dataset.select(range(min(2000, len(dataset))))  # Subset for GRPO
print(f"GRPO training set: {len(grpo_train)} problems")

In [ ]:
# Cell 6: Train GRPO
if trainer is not None and model is not None:
    print("Starting GRPO training...")
    print(f"Group size G={config.GROUP_SIZE}, KL penalty={config.KL_PENALTY}")
    
    # We rewrite prompts format for GRPOTrainer expectation
    # GRPOTrainer expects 'prompt' field
    grpo_dataset = grpo_train.map(lambda x: {"prompt": x["question"], "ground_truth": x["answer"]})
    
    result = trainer.train(
        train_dataset=grpo_dataset,
        reward_function=reward_fn,
    )
    
    trainer.save_adapter("checkpoints/grpo/final_adapter")
else:
    print("GRPO training skipped (no model loaded).")

In [ ]:
# Cell 7: Sync to Hugging Face Hub
from scripts.sync_to_hub import sync_adapter

api_token = os.environ.get("HF_TOKEN")
if api_token and model is not None:
    print("Syncing GRPO adapter to Hugging Face Hub...")
    sync_adapter(
        adapter_path="checkpoints/grpo/final_adapter",
        repo_id="samar/atrd-nemotron-grpo-r32",
        commit_message="GRPO Phase 3: RL-optimized policy after 1 epoch",
        private=True,
    )
else:
    print("Skipping Hugging Face Sync (HF_TOKEN not set or model is None).")

In [ ]:
# Cell 8: Cleanup
import gc
if 'model' in globals() and model is not None:
    del model, base_model, trainer
torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared.")
print('P3 Complete — Phase gate: python scripts/verify_unit_completion.py P3 grpo')